# Rule composition across engines — a professional-network demo

This notebook builds **three base rules** over a small professional network, then
**composes** them into **three higher-level rules** using `&` (AND), `|` (OR) and
port joins. The same composed logic is then run across several reasoning engines so
you can compare how each one *explains* a conclusion.

**The rules (the `IF … THEN` shapes this demo implements)**

Base rules — the reusable bricks:

| Rule | IF | THEN |
|---|---|---|
| `colleagues(A, B)` | A works at company C **and** B works at company C **and** A ≠ B | A and B are colleagues |
| `works_on_project(X, P)` | X is assigned to project P **and** P is active **and** X's workload on P > 20h | X works on project P |
| `core_expertise(Y)` | Y holds skill S **and** S is a core skill | Y has core expertise |

Composite rules — built from the bricks:

| Rule | Definition |
|---|---|
| `teammates(A, B)` | `colleagues(A,B)` ∧ (A works on a project ∧ B works on the **same** project) |
| `active_professional_peers(A, B)` | `colleagues(A,B)` ∧ (A works on a project ∨ A has core expertise) ∧ (B works on a project ∨ B has core expertise) |
| `confirmed_collaboration_link(A, B)` | `colleagues(A,B)` ∧ ((A works on a project ∧ A has core expertise) ∨ (B works on a project ∧ B has core expertise)) |

**Engines covered**

| Engine | What it adds to the explanation |
|---|---|
| `native` | deterministic boolean reasoning chain (the baseline) |
| `problog` | the same chain **plus a probability** propagated from fact-level uncertainty |
| `souffle` | high-performance Datalog (handles the datalog-shaped base rules) |
| `pyreason` | a *different model*: interval-valued, **temporal** graph propagation (shown on its own terms) |


## 0 · Setup

Locate `src/` so the notebook runs from either the repo root or `examples/`, and register the probabilistic engine.

In [1]:
import sys, pathlib
_root = pathlib.Path.cwd()
while not (_root / "src" / "factgraph").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root / "src"))

import factgraph.adapters.problog  # noqa: F401  (registers the "problog" engine)
from factgraph.sdk import (
    Entity, FactGraph, Field, Identity, Rule,
    build_application_rule, vars, ProbLogConfig,
)
from factgraph.core.evidence.write_protocol import set_field

# ProbLog config: read each fact's (raw_kind, bound) meta and use the degenerate
# interval [p, p] as a point probability p.
PROBLOG = ProbLogConfig(
    name="demo",
    uncertainty_projection={
        "probabilistic": {"policy": "identity_probability"},
        "possibilistic": {"policy": "reject"},
        "fallback": "use_default",
    },
)
print("ready")

ready


## 1 · Schema — a small professional network

Six entity types. Each declares a stable `Identity`, its `Field`s, and `repr`
templates so explanations read in plain language.

| Entity | Fields | Atom it supports |
|---|---|---|
| `Company` | — | *A works at company C* |
| `Project` | `active: bool` | *project P is active* |
| `Skill` | `is_core: bool` | *skill S is a core skill* |
| `User` | `company: Company` | *A works at company C* |
| `Assignment` | `user, project, workload: int` | *X assigned to P* + *workload > 20* |
| `Holding` | `user, skill` | *Y holds skill S* |


In [2]:
class Company(Entity):
    company_id: str = Identity(repr="%ENT id %FLD")
    class Meta: repr = "Company %company_id"

class Project(Entity):
    project_id: str = Identity(repr="%ENT id %FLD")
    active: bool = Field(repr="%ENT active %FLD")
    class Meta: repr = "Project %project_id"

class Skill(Entity):
    skill_id: str = Identity(repr="%ENT id %FLD")
    is_core: bool = Field(repr="%ENT core %FLD")
    class Meta: repr = "Skill %skill_id"

class User(Entity):
    user_id: str = Identity(repr="%ENT id %FLD")
    company: Company = Field(repr="%ENT works at %FLD")
    class Meta: repr = "User %user_id"

class Assignment(Entity):
    assignment_id: str = Identity(repr="%ENT id %FLD")
    user: User = Field(repr="%ENT by %FLD")
    project: Project = Field(repr="%ENT on %FLD")
    workload: int = Field(repr="%ENT workload %FLD h")
    class Meta: repr = "Assignment %assignment_id"

class Holding(Entity):
    holding_id: str = Identity(repr="%ENT id %FLD")
    user: User = Field(repr="%ENT by %FLD")
    skill: Skill = Field(repr="%ENT of %FLD")
    class Meta: repr = "Holding %holding_id"

fg = FactGraph.create(schema_classes=[Company, Project, Skill, User, Assignment, Holding])
print("schema loaded")

schema loaded


## 2 · Facts — carrying probability and validity meta

Every write can attach `meta`. Two keys matter here:

- **Uncertainty** — `meta={"raw_kind": "probabilistic", "bound": [lo, hi]}`. The pair
  must appear together. ProbLog reads the degenerate interval `[p, p]` as the point
  probability `p`; native/souffle ignore it.
- **Validity** — `meta={"valid_from": "...", "valid_to": "..."}` (ISO business time).

The cast (named people, so the explanations read naturally):

- **Alice** — at *ACME* (p=0.95), works *P1* 30h, holds *Python* (a core skill).
- **Bob** — at *ACME* (p=0.90), works *P1* 25h, no recorded skill.
- **Carol** — at *ACME* (p=0.90), **no project assignment** — a colleague who is *not* a teammate.
- *P1* active (p=0.80); *Python* is core (p=0.85).

`Carol` exists so you can watch composition **narrow** the result set (everyone is a
colleague; only some are teammates) — and later, so we can ask *why* she is not a teammate.


In [3]:
def ex(e, t):
    """Emit the <Entity>:exists claim (fg.entities.create does not emit it yet)."""
    set_field(fg.ledger, f"{t}:exists", e, [], None)

def prob(p, **extra):
    return {"raw_kind": "probabilistic", "bound": [p, p], **extra}

acme = fg.entities.create(Company, company_id="ACME"); ex(acme, "Company")

alice = fg.entities.create(User, user_id="Alice")
fg.fields.set(User.company, alice, acme, meta=prob(0.95, valid_from="2026-01-01")); ex(alice, "User")
bob = fg.entities.create(User, user_id="Bob")
fg.fields.set(User.company, bob, acme, meta=prob(0.90, valid_from="2026-01-01")); ex(bob, "User")
carol = fg.entities.create(User, user_id="Carol")
fg.fields.set(User.company, carol, acme, meta=prob(0.90, valid_from="2026-01-01")); ex(carol, "User")

p1 = fg.entities.create(Project, project_id="P1")
fg.fields.set(Project.active, p1, True, meta=prob(0.80)); ex(p1, "Project")   # P1 is active

for name, u, wl in [("Alice-P1", alice, 30), ("Bob-P1", bob, 25)]:   # Alice and Bob work P1; Carol does not
    g = fg.entities.create(Assignment, assignment_id=name)
    fg.fields.set(Assignment.user, g, u)
    fg.fields.set(Assignment.project, g, p1)
    fg.fields.set(Assignment.workload, g, wl); ex(g, "Assignment")

py = fg.entities.create(Skill, skill_id="Python")
fg.fields.set(Skill.is_core, py, True, meta=prob(0.85)); ex(py, "Skill")
h = fg.entities.create(Holding, holding_id="H-Alice")
fg.fields.set(Holding.user, h, alice); fg.fields.set(Holding.skill, h, py); ex(h, "Holding")
h = fg.entities.create(Holding, holding_id="H-Carol")
fg.fields.set(Holding.user, h, carol); fg.fields.set(Holding.skill, h, py); ex(h, "Holding")
print("facts written")

facts written


## 3 · Three base rules — the reusable bricks

A `Rule` declares a `when` body and a `ports` head. Note `works_on_project`: the
Entity-DSL only sugars `==`, so to filter `workload > 20` we **bind the field to a
logic variable first** (`Assignment(g).workload == wl`) and then compare `wl > 20`.


In [4]:
with vars("a", "b", "c") as (a, b, c):
    colleagues = build_application_rule(
        id="colleagues", version="v1",
        when=[User(a), User(b), Company(c),
              User(a).company == c, User(b).company == c, a != b],
        ports={"A": a, "B": b},
        repr="%A and %B are colleagues",
    )

with vars("x", "p", "g", "wl") as (x, p, g, wl):
    works_on_project = build_application_rule(
        id="works_on_project", version="v1",
        when=[User(x), Project(p), Assignment(g),
              Assignment(g).user == x, Assignment(g).project == p,
              Project(p).active == True,
              Assignment(g).workload == wl, wl > 20],   # bind, then compare
        ports={"X": x, "P": p},
        repr="%X works on project %P",
    )

with vars("y", "s", "hh") as (y, s, hh):
    core_expertise = build_application_rule(
        id="core_expertise", version="v1",
        when=[User(y), Skill(s), Holding(hh),
              Holding(hh).user == y, Holding(hh).skill == s, Skill(s).is_core == True],
        ports={"Y": y},
        repr="%Y has core expertise",
    )
print("base rules built")

base rules built


## 4 · Run a base rule and read its explanation

`fg.eval.evaluate(rule, head=rule, engine=...)` returns rows; `row.explain().narrate()` renders the reasoning chain. (`colleagues` yields ordered pairs, so Alice↔Bob counts twice.)

In [5]:
res = fg.eval.evaluate(colleagues, head=colleagues, engine="native")
print(f"colleagues — {res.count()} rows\n")
for row in res:
    for line in row.explain().narrate():
        print(line)
    break

colleagues — 6 rows

Conclusion ── User Alice and User Bob are colleagues
              [colleagues · run_v1:33bf14a0… · c0]  holds
      produces:  A = User Alice,  B = User Bob
  colleagues  [holds]
       ✓ User Alice exists  [c0:atom:0]  holds
       ✓ User Bob exists  [c0:atom:1]  holds
       ✓ Company ACME exists  [c0:atom:2]  holds
       ✓ User Alice works at Company ACME  [c0:atom:3]  holds
       ✓ User Bob works at Company ACME  [c0:atom:4]  holds
       ✓ User Alice does not equal User Bob  [c0:atom:5]  holds


## 5 · The same rule on three engines

`works_on_project` is datalog-shaped, so all three SDK engines run it. Watch the
`certainty`: native/souffle are boolean `1.0`; **problog** carries a real
probability derived from the fact meta (`active@0.80` flows through).

> **Engine note.** `souffle` shines on datalog-shaped rules but rejects
> `colleagues` (it pairs two `User` variables → a *witness binding conflict*) and
> large composites (its witness relation arity is capped at 22). `pyreason` is a
> different model entirely — see §7.


In [6]:
def cross_engine(rule, label):
    print(f"### {label}")
    for name, kw in [("native", {"engine": "native"}),
                     ("souffle", {"engine": "souffle"}),
                     ("problog", {"config": PROBLOG})]:
        try:
            r = fg.eval.evaluate(rule, head=rule, **kw)
            c = r.first().certainty
            print(f"  {name:8} -> {r.count()} rows | certainty={c.lo} ({c.kind})")
        except Exception as e:
            print(f"  {name:8} -> {type(e).__name__}: {str(e).splitlines()[0][:70]}")

cross_engine(works_on_project, "works_on_project")
cross_engine(core_expertise, "core_expertise")
print()
# colleagues: native/problog handle it, souffle does not
print("### colleagues across engines")
cross_engine(colleagues, "colleagues")

### works_on_project
  native   -> 2 rows | certainty=1.0 (boolean)
  souffle  -> 2 rows | certainty=1.0 (boolean)
  problog  -> 2 rows | certainty=0.8 (probabilistic)
### core_expertise
  native   -> 2 rows | certainty=1.0 (boolean)
  souffle  -> 2 rows | certainty=1.0 (boolean)
  problog  -> 2 rows | certainty=0.85 (probabilistic)

### colleagues across engines
### colleagues
  native   -> 6 rows | certainty=1.0 (boolean)
  souffle  -> WhereValidationError: witness fact binding conflict for $colleagues__b
  problog  -> 6 rows | certainty=0.855 (probabilistic)


A problog narrate shows the *same* chain as native, annotated with per-atom and aggregated probabilities:

In [7]:
row = fg.eval.evaluate(works_on_project, head=works_on_project, config=PROBLOG).first()
print(f"certainty = {row.certainty.lo} ({row.certainty.kind})\n")
for line in row.explain().narrate():
    print(line)

certainty = 0.8 (probabilistic)

Conclusion ── User Bob works on project Project P1
              [works_on_project · run_v1:044ce265… · c0]  holds with probability 0.8
      produces:  X = User Bob,  P = Project P1
  works_on_project  [holds]  (p = 1 × 1 × 1 × 1 × 1 × 0.8 × 1 × 1 = 0.8)
       ✓ User Bob exists  [c0:atom:0]  holds
       ✓ Project P1 exists  [c0:atom:1]  holds
       ✓ Assignment Bob-P1 exists  [c0:atom:2]  holds
       ✓ Assignment Bob-P1 by User Bob  [c0:atom:3]  holds
       ✓ Assignment Bob-P1 on Project P1  [c0:atom:4]  holds
       ✓ Project P1 active True  [c0:atom:5]  holds (p = 0.8)
       ✓ Assignment Bob-P1 workload 25 h  [c0:atom:6]  holds
       ✓ 25 > 20  [c0:atom:7]  holds


## 6 · Composition — from bricks to higher-level rules

`RuleExpr` glues rules together:

- `r1 & r2` (AND), `r1 | r2` (OR)
- `rule.as_("alias")` — a named *occurrence* (required when a rule appears more than once)
- `(g).join(left.port("X").eq(right.port("Y")))` — an equality join across occurrences
- evaluate with `head=<a Rule whose ports name the output>` — here a tiny rule that
  just exposes `A`/`B` and carries the conclusion `repr`.

### The port-namespace nuance (why some rules get *per-side variants*)

A composite's ports live in one flat namespace. If you apply `works_on_project`
(port `X`) to **both** colleagues inside a single conjunction, the composite sees
two `X` ports bound to different users → *"declared port 'X' is ambiguous"*. Two ways out:

1. **Separate OR branches** — if the two uses live in *different* branches of an `|`,
   no clash. `confirmed_collaboration_link` is shaped this way, so it reuses the base
   rules directly.
2. **Per-side variants** — when both uses sit in the *same* conjunction
   (`teammates`, `active_professional_peers`), instantiate the brick once per side
   with **distinct port names** (`Ua/Pa` vs `Ub/Pb`).


In [8]:
# Per-side variants: the same logic as the base bricks, with side-specific port names.
def works_for(side):                       # side = "a" | "b"
    with vars(f"x{side}", f"p{side}", f"g{side}", f"wl{side}") as (x, p, g, wl):
        return build_application_rule(
            id=f"works_{side}", version="v1",
            when=[User(x), Project(p), Assignment(g),
                  Assignment(g).user == x, Assignment(g).project == p,
                  Project(p).active == True,
                  Assignment(g).workload == wl, wl > 20],
            ports={f"U{side}": x, f"P{side}": p},
            repr=f"%U{side} works on project %P{side}",
        )

def core_for(side):
    with vars(f"y{side}", f"s{side}", f"h{side}") as (y, s, hh):
        return build_application_rule(
            id=f"core_{side}", version="v1",
            when=[User(y), Skill(s), Holding(hh),
                  Holding(hh).user == y, Holding(hh).skill == s, Skill(s).is_core == True],
            ports={f"Y{side}": y},
            repr=f"%Y{side} has core expertise",
        )

works_a, works_b = works_for("a"), works_for("b")
core_a,  core_b  = core_for("a"),  core_for("b")

# Conclusion heads: tiny rules that expose A/B and name the conclusion.
with vars("ha", "hb") as (ha, hb):
    teammates_head = build_application_rule(
        id="teammates", version="v1", when=[User(ha), User(hb)],
        ports={"A": ha, "B": hb}, repr="%A and %B are teammates")
    peers_head = build_application_rule(
        id="active_professional_peers", version="v1", when=[User(ha), User(hb)],
        ports={"A": ha, "B": hb}, repr="%A and %B are active professional peers")
    collab_head = build_application_rule(
        id="confirmed_collaboration_link", version="v1", when=[User(ha), User(hb)],
        ports={"A": ha, "B": hb}, repr="%A and %B have a confirmed collaboration link")
print("variants + heads built")

variants + heads built


### 6a · `teammates` — pure AND

`colleagues(A,B)` ∧ A works ∧ B works, joined so both works land on the **same**
project (`Pa = Pb`) and on the right colleague (`A = Ua`, `B = Ub`). A pure
conjunction, so the whole proof tree binds and reads cleanly.

> The tree is narrated under `native` (which renders every occurrence's label);
> `problog` reports the propagated **certainty** shown above (`0.684`). Watch the
> probability flow: `colleagues@0.855 × works_a × works_b → 0.684`.


In [9]:
col = colleagues.as_("col")
wa, wb = works_a.as_("wa"), works_b.as_("wb")
ca, cb = core_a.as_("ca"), core_b.as_("cb")

# reusable join fragments
A_works = col.port("A").eq(wa.port("Ua"))
A_core  = col.port("A").eq(ca.port("Ya"))
B_works = col.port("B").eq(wb.port("Ub"))
B_core  = col.port("B").eq(cb.port("Yb"))

teammates = (col & wa & wb) \
    .join(A_works).join(B_works) \
    .join(wa.port("Pa").eq(wb.port("Pb")))          # same project

tm_native = fg.eval.evaluate(teammates, head=teammates_head, engine="native")
col_n     = fg.eval.evaluate(colleagues, head=colleagues, engine="native").count()
tm_prob   = fg.eval.evaluate(teammates, head=teammates_head, config=PROBLOG).first().certainty
print(f"colleagues={col_n} rows  ->  teammates={tm_native.count()} rows   (composition narrows the set)")
print(f"problog certainty for a teammates conclusion: {tm_prob.lo}\n")

# Narrate under native: it renders every occurrence's label. problog carries the
# same tree plus the probability printed above.
for row in tm_native:
    for line in row.explain().narrate():
        print(line)

colleagues=6 rows  ->  teammates=2 rows   (composition narrows the set)
problog certainty for a teammates conclusion: 0.684

Conclusion ── User Bob and User Alice are teammates
              [teammates · run_v1:2fb49b42… · c0]  holds
      produces:  A = User Bob,  B = User Alice
  Derivation:  teammates <= ( colleagues AND works_a AND works_b )
               join:  colleagues.A = works_a.Ua  [holds]
               join:  colleagues.B = works_b.Ub  [holds]
               join:  works_a.Pa = works_b.Pb  [holds]
  teammates [head] ── "User Bob and User Alice are teammates"  [holds]
       ✓ User Bob exists  [c0:atom:22]  holds
       ✓ User Alice exists  [c0:atom:23]  holds
  colleagues ── "User Bob and User Alice are colleagues"  [holds]
       ✓ User Bob exists  [c0:atom:0]  holds
       ✓ User Alice exists  [c0:atom:1]  holds
       ✓ Company ACME exists  [c0:atom:2]  holds
       ✓ User Bob works at Company ACME  [c0:atom:3]  holds
       ✓ User Alice works at Company ACME  [c0:atom

### 6b · `active_professional_peers` — AND of ORs

`colleagues(A,B)` ∧ (A works ∨ A has core expertise) ∧ (B works ∨ B has core
expertise). Each side is satisfied by *either* brick, so the join fragments are
attached and the normalizer distributes them across the OR branches.


In [10]:
peers = (col & (wa | ca) & (wb | cb)) \
    .join(A_works).join(A_core).join(B_works).join(B_core)

for name, kw in [("native", {"engine": "native"}), ("problog", {"config": PROBLOG})]:
    r = fg.eval.evaluate(peers, head=peers_head, **kw)
    extra = "" if name == "native" else f" | certainty={r.first().certainty.lo}"
    print(f"active_professional_peers [{name}] -> {r.count()} rows{extra}")

active_professional_peers [native] -> 6 rows
active_professional_peers [problog] -> 6 rows | certainty=0.5508


In [11]:
res_psadas = fg.eval.evaluate(peers, head=peers_head, config=PROBLOG).first()
for line in res_psadas.explain().narrate():
    print(line)

Conclusion ── User Carol and User Bob are active professional peers
              [active_professional_peers · run_v1:0a1312a8… · c0|c1|c2|c3 (concluded via c2)]  holds with probability 0.5508
      produces:  A = User Carol,  B = User Bob
  Derivation:  active_professional_peers <= ( colleagues AND works_a AND works_b ) OR ( colleagues AND core_b AND works_a ) OR ( colleagues AND core_a AND works_b ) OR ( colleagues AND core_a AND core_b )
      probability:  0.5508
▸ Path c0  [fails]
               join:  colleagues.A = works_a.Ua  [not_reached]
               join:  colleagues.B = works_b.Ub  [not_reached]
  active_professional_peers [head] ── "User Carol and User Bob are active professional peers"  [not_reached]
       ○ User:exists not reached  [c0:atom:22]  not_reached
       ○ User:exists not reached  [c0:atom:23]  not_reached
  colleagues ── "User Carol and User Bob are colleagues"  [holds]
       ✓ User Carol exists  [c0:atom:0]  holds
       ✓ User Bob exists  [c0:atom:1]  ho

### 6c · `confirmed_collaboration_link` — AND of (AND-OR)

`colleagues(A,B)` ∧ ((A works ∧ A core) ∨ (B works ∧ B core)). The two
alternatives are in **separate OR branches**, so this one reuses the base
`works_on_project` / `core_expertise` directly (no variants needed).

> In an OR composite, the narrate prints every branch; the branch that did **not**
> fire shows unbound ports like `<X>` — that is the alternative path, shown for
> completeness, not an error.


In [12]:
cl = colleagues.as_("col")
owa, owb = works_on_project.as_("wa"), works_on_project.as_("wb")
oca, ocb = core_expertise.as_("ca"), core_expertise.as_("cb")

collab = (cl & ((owa & oca) | (owb & ocb))) \
    .join(cl.port("A").eq(owa.port("X"))).join(cl.port("A").eq(oca.port("Y"))) \
    .join(cl.port("B").eq(owb.port("X"))).join(cl.port("B").eq(ocb.port("Y")))

r_native = fg.eval.evaluate(collab, head=collab_head, engine="native")
r_prob   = fg.eval.evaluate(collab, head=collab_head, config=PROBLOG)
print(f"confirmed_collaboration_link -> {r_native.count()} rows  "
      f"(problog certainty = {r_prob.first().certainty.lo})\n")
for row in r_native:
    for line in row.explain().narrate():
        print(line)
    break

confirmed_collaboration_link -> 4 rows  (problog certainty = 0.5814)

Conclusion ── User Carol and User Alice have a confirmed collaboration link
              [confirmed_collaboration_link · run_v1:0d6a80df… · c0|c1 (concluded via c1)]  holds
      produces:  A = User Carol,  B = User Alice
  Derivation:  confirmed_collaboration_link <= ( colleagues AND core_expertise AND works_on_project ) OR ( colleagues AND core_expertise AND works_on_project )
▸ Path c0  [fails]
               join:  colleagues.A = core_expertise.Y  [holds]
               join:  colleagues.A = works_on_project.X  [holds]
  confirmed_collaboration_link [head] ── "User Carol and User Alice have a confirmed collaboration link"  [holds]
       ✓ User Carol exists  [c0:atom:20]  holds
       ✓ User Alice exists  [c0:atom:21]  holds
  core_expertise ── "<Y> has core expertise"  [holds]
       ✓ User Carol exists  [c0:atom:6]  holds
       ✓ Skill Python exists  [c0:atom:7]  holds
       ✓ Holding H-Carol exists  [c0:ato

## 7 · Why a conclusion does NOT hold — the explain-head technique

`fg.eval.evaluate(...)` returns only the rows that **hold**. A pair that is *not* a
teammate produces no row — so there is nothing to call `.explain()` on. To explain a
**non-holding** conclusion you construct a closed *explain head*: a `Rule` whose body
**pins the subject(s) to specific entities** (the identity pin goes *first*, so the
prober binds that exact entity before checking the rest). Passed to
`fg.eval.explain(...)`, a non-holding conclusion returns
`Explanation(status="failed", failure_class="closed_head_false")` — still carrying a
full failure tree you can `.narrate()`.

The narrate marks each atom: `✓` holds · `✗` present but the check failed (the real
value is shown) · `○` a required fact was entirely absent (unbound).

We give two of Alice's colleagues fact-level gaps and ask *why* they are not her teammates:

- **Carol** — no project assignment at all (a **missing fact**).
- **Dave** — assigned to P1, but for only 15h, under the `> 20` threshold (a **failed check**).


In [13]:
# Carol (already in the cast) has no project. Add Dave — assigned to P1 for only 15h.
dave = fg.entities.create(User, user_id="Dave"); fg.fields.set(User.company, dave, acme); ex(dave, "User")
gd = fg.entities.create(Assignment, assignment_id="Dave-P1")
fg.fields.set(Assignment.user, gd, dave); fg.fields.set(Assignment.project, gd, p1)
fg.fields.set(Assignment.workload, gd, 15); ex(gd, "Assignment")

n = fg.eval.evaluate(teammates, head=teammates_head, engine="native").count()
print(f"teammates rows: {n}  — only Alice & Bob qualify; "
      f"Alice-Carol and Alice-Dave produce no row to explain.")

def works_explain_head(user_id):
    """A CLOSED explain head = works_on_project's body + an identity pin placed FIRST,
    so the prober binds that exact user before checking the rest of the body."""
    with vars("x", "p", "g", "wl") as (x, p, g, wl):
        return build_application_rule(
            id=f"works_check_{user_id}",
            when=[User(x), User(x).user_id == user_id,            # pin the subject FIRST
                  Project(p), Project(p).project_id == "P1",
                  Assignment(g), Assignment(g).user == x, Assignment(g).project == p,
                  Project(p).active == True, Assignment(g).workload == wl, wl > 20],
            ports={"X": x, "P": p}, repr="%X works on project %P")

teammates rows: 2  — only Alice & Bob qualify; Alice-Carol and Alice-Dave produce no row to explain.


### 7a · The reason — point an explain head at the failing requirement

`teammates(A, B)` needs *both* colleagues to satisfy `works_on_project`. Carol and
Dave each break that requirement, so they never appear in a `teammates` row. To see
*why*, build a closed explain head for `works_on_project`, pinned to the colleague
(identity pin **first**, so the prober binds that exact person). The head carries the
rule body, so `fg.eval.explain` returns `status="failed"` and the failure tree pins
the **exact** atom:

- **Carol** → `✗ … by User Carol`: no `Assignment` belongs to her (a missing fact).
- **Dave** → `✗ 15 > 20`: the assignment exists, but the workload check fails on the real value.


In [21]:
for who in ("Carol", "Dave"):
    head = works_explain_head(who)
    exp = fg.eval.explain(head, head=head, engine="native")
    print(f"########## works_on_project({who}) — status={exp.status} ##########")
    for line in exp.narrate():
        print(line)
    print()

########## works_on_project(Carol) — status=failed ##########
NOT concluded ── User Carol works on project Project P1 (closed_head_false)
              [works_check_Carol · evalr_v1:ccb9592fe5e8e1352d9732465edea4d5b4a7ff5350ec5273942214f8938076fc · c0]  fails
      produces:  X = User Carol,  P = Project P1
  works_check_Carol ── "<X> works on project <P>"  [fails]
       ✓ User Carol exists  [c0:atom:0]  holds
       ✓ User Carol id Carol  [c0:atom:1]  holds
       ✓ Project P1 exists  [c0:atom:2]  holds
       ✓ Project P1 id P1  [c0:atom:3]  holds
       ✓ Assignment Bob-P1 exists  [c0:atom:4]  holds
       ✗ Assignment Bob-P1 by User Carol  [c0:atom:5]  fails
       ✓ Assignment Bob-P1 on Project P1  [c0:atom:6]  holds
       ✓ Project P1 active True  [c0:atom:7]  holds
       ✓ Assignment Bob-P1 workload 25 h  [c0:atom:8]  holds
       ✓ 25 > 20  [c0:atom:9]  holds

########## works_on_project(Dave) — status=failed ##########
NOT concluded ── User Dave works on project Project P1 

### 7b · The composite verdict — the **full body**, expanded

Pin `teammates` to the pair `(Alice, Carol)`. `fg.eval.explain` returns
`status="failed"` (`closed_head_false`): the pair is not a teammate.

The failure tree now **expands the entire composite body** — not just the head's
existence closure. Every occurrence (`colleagues`, `works_a`, `works_b`), every
atom, and every cross-occurrence **join** carries one of the three verdicts
(`✓` holds · `✗` fails · `○` not reached) — *全量*, nothing omitted. Read the tree:

- `colleagues` **holds** — Alice and Carol are colleagues.
- `works_a` **holds** — Alice is assigned to the active project P1 at 30h (> 20).
- `works_b` **fails** at `✗ Assignment … by User Carol` — Carol has **no
  assignment**, so the membership check for her side fails on the exact missing
  fact (the same culprit 7a isolates at the brick level).
- the joins all **hold** — both colleague links and the same-project link
  `works_a.Pa = works_b.Pb`; only `works_b`'s own atom breaks the chain.

The verdict is **structural reachability** (a non-holding conclusion carries no
derived probability), and the cell runs it on both `native` and `problog`. One
per-engine nuance: after the failing atom, `native` keeps reporting each remaining
atom's own value (so some still read `✓`), while the reach engines mark the tail
`○ not reached`; the **culprit is identical** on both. This is 7a's recipe lifted
to the composite — a closed head over the whole rule expression verdicts the
**occurrence + join** structure end to end.

> A large composite like `teammates` exceeds Souffle's witness-arity cap
> (arity > 22), so its closed-head explain runs on `native` / `problog`; pointing
> `engine="souffle"` at it raises a capacity error (not a wrong answer).

In [15]:
with vars("a", "b") as (a, b):
    teammates_alice_carol = build_application_rule(
        id="teammates_head",
        when=[User(a), User(a).user_id == "Alice", User(b), User(b).user_id == "Carol"],
        ports={"A": a, "B": b}, repr="%A and %B are teammates")

# The closed head pins (Alice, Carol); the failure explain now expands the FULL
# composite body. Same structural tree on native and problog (problog would add
# probabilities to a *holding* tree; a non-holding one is structural).
for engine_kw, label in [(dict(engine="native"), "native"),
                         (dict(engine="problog", config=PROBLOG), "problog")]:
    exp = fg.eval.explain(teammates, head=teammates_alice_carol, **engine_kw)
    print(f"########## teammates(Alice, Carol) [{label}] — "
          f"status={exp.status} | {exp.failure_class} ##########")
    for line in exp.narrate():
        print(line)
    print()


########## teammates(Alice, Carol) [native] — status=failed | closed_head_false ##########
NOT concluded ── User Alice and User Carol are teammates (closed_head_false)
              [teammates_head · evalr_v1:c71c7090e00ab7bdfc953ea94249d2fbfa817df9cd0464d4f2f3cac899fc52e2 · c0]  fails
      produces:  A = User Alice,  B = User Carol
  Derivation:  teammates_head <= ( colleagues AND works_a AND works_b )
               join:  colleagues.A = works_a.Ua  [holds]
               join:  colleagues.B = works_b.Ub  [holds]
               join:  works_a.Pa = works_b.Pb  [holds]
  teammates_head [head] ── "User Alice and User Carol are teammates"  [holds]
       ✓ User Alice exists  [c0:atom:22]  holds
       ✓ User Alice id Alice  [c0:atom:23]  holds
       ✓ User Carol exists  [c0:atom:24]  holds
       ✓ User Carol id Carol  [c0:atom:25]  holds
  colleagues ── "colleagues"  [holds]
       ✓ User Alice exists  [c0:atom:0]  holds
       ✓ User Carol exists  [c0:atom:1]  holds
       ✓ Company 

In [16]:
from pprint import pp
pp(exp)

Explanation(status='failed',
            evidence=EvidenceGraph(graph_id='evalr_v1:5956433ecc826aca299ab539f52d1de4b5c8879386e43d5de741002d13b050dd:closed_head_false',
                                   engine='problog',
                                   layout_hint='tree',
                                   subject_binding=mappingproxy({'A': 'User '
                                                                      'Alice',
                                                                 'B': 'User '
                                                                      'Carol'}),
                                   paths=(EvidenceTree(tree_id='c0',
                                                       status='fails',
                                                       rules=(EvidenceRule(occurrence_alias='teammates_head',
                                                                           rule_id='teammates_head',
                                                         

### 7c · A *combination* verdicts every route — sub-rule, atom, **and** join

`teammate_check = colleagues(A, B) ∧ (B works on a project ∨ B has core expertise)` —
three existing bricks combined (no logic re-written). For **Alice & Bob** it
**holds** via route c0 (Bob works P1). The OR's other route (c1, core expertise)
**fails**, and the combination shows *exactly why* — at every altitude: per
sub-rule, per atom, and per **join**.

- route **c0** `colleagues ∧ works_b` — **holds**: Bob is on active P1 at 25h, and
  the join `colleagues.B = works_b.Ub` ties that work to Bob.
- route **c1** `colleagues ∧ core_b` — **fails** at `✗ Holding H-Carol by User Bob`.
  The join `colleagues.B = core_b.Yb` **holds** (it correctly scopes `core_b` to
  **Bob**); `core_b` then looks for a core-skill holding *owned by Bob* and finds
  none — the only Python holding in scope is Carol's, so `holding:user(…, Bob)`
  fails. Bob simply has no recorded skill.

Read it this way: a combination reports its verdict at every level — each route's
sub-rules, each atom, and each **join** that ties the occurrences to the same
people. Because the explain seed is propagated across those joins, every
occurrence binds to the right person, so a route's failure is pinned to the
**precise atom** (here: Bob owns no core holding) rather than deflected to an
unbound join. It is 7a's atom-level precision, holding across an OR combination.

In [17]:
# teammate_check reuses three existing bricks — colleagues + the per-side works_b / core_b variants.
tc_col = colleagues.as_("col"); tc_wb = works_b.as_("wb"); tc_cb = core_b.as_("cb")
teammate_check = (tc_col & (tc_wb | tc_cb)) \
    .join(tc_col.port("B").eq(tc_wb.port("Ub"))) \
    .join(tc_col.port("B").eq(tc_cb.port("Yb")))

with vars("ha", "hb") as (ha, hb):
    teammate_check_head = build_application_rule(
        id="teammate_check", when=[User(ha), User(hb)],
        ports={"A": ha, "B": hb}, repr="%A and %B")

alice_ref = fg.entities.ref(User, user_id="Alice")
bob_ref   = fg.entities.ref(User, user_id="Bob")
for row in fg.eval.evaluate(teammate_check, head=teammate_check_head, engine="native"):
    if row.bindings["A"]["value"] == alice_ref and row.bindings["B"]["value"] == bob_ref:
        for line in row.explain().narrate():
            print(line)
        break

Conclusion ── User Alice and User Bob
              [teammate_check · run_v1:7cfeaa86… · c0|c1 (concluded via c0)]  holds
      produces:  A = User Alice,  B = User Bob
  Derivation:  teammate_check <= ( colleagues AND works_b ) OR ( colleagues AND core_b )
▸ Path c0  [holds]
               join:  colleagues.B = works_b.Ub  [holds]
  teammate_check [head] ── "User Alice and User Bob"  [holds]
       ✓ User Alice exists  [c0:atom:14]  holds
       ✓ User Bob exists  [c0:atom:15]  holds
  colleagues ── "User Alice and User Bob are colleagues"  [holds]
       ✓ User Alice exists  [c0:atom:0]  holds
       ✓ User Bob exists  [c0:atom:1]  holds
       ✓ Company ACME exists  [c0:atom:2]  holds
       ✓ User Alice works at Company ACME  [c0:atom:3]  holds
       ✓ User Bob works at Company ACME  [c0:atom:4]  holds
       ✓ User Alice does not equal User Bob  [c0:atom:5]  holds
  works_b ── "User Bob works on project Project P1"  [holds]
       ✓ User Bob exists  [c0:atom:6]  holds
       ✓ Pr

> **Engine note.** The failure tree has the same shape across `native` / `problog` /
> `souffle` for rules within Souffle's witness-arity cap
> (`tests/test_explain_cross_engine_conformance.py`). A large composite like
> `teammates` exceeds that cap (arity > 22), so its explain runs on `native` — pointing
> `engine="souffle"` at it raises `WhereValidationError` (a capacity limit, not a wrong answer).
>
> The recipe generalizes: to ask *"why is X **not** a teammate / peer / collaborator?"*
> pin the subjects in a closed head and point it at the brick that fails.


## 8 · `pyreason` — a fundamentally different model

PyReason is not a drop-in fourth engine for these rules. It performs **interval-valued,
temporal reasoning over a graph** (`head(x) <-T body(y), edge(x,y)`): values are
intervals `[lo, hi]`, the world is *open* (a missing fact is `[0,1]`, not false), and
conclusions are stamped with a **timestep**.

Two consequences make the SDK `evaluate(...)` surface reject the rules above:

1. its WHERE compiler only accepts plain existence atoms — `eq` / `ne` / `>` / `bool`
   all raise; and
2. it cannot register fact components that contain `:` — but every entity reference
   here is a canonical `idref_v1:User:…` token.

The cell below shows that boundary directly:


In [18]:
try:
    fg.eval.evaluate(teammates, head=teammates_head, engine="pyreason")
except Exception as e:
    print(f"pyreason via SDK evaluate -> {type(e).__name__}:\n  {str(e).splitlines()[0]}")

pyreason via SDK evaluate -> SDKStoreError:
  evaluate(rule_expr, ...) unsupported for engine='pyreason': unsupported feature 'ne' from source-rule-grammar; supported alternative engines: native, souffle, problog


### 8a · PyReason on its own terms — influence propagation

To actually *see* a PyReason explanation we drop to its adapter-internal runner
(`PyReasonSession` + `run_pyreason`), which uses raw string node ids. The toy graph:
**influence spreads through collaboration** — `Alice` is influential; the rule
`influential(x) <-1 influential(y), collab(y, x)` pushes it one hop per timestep.

The output shows the three things souffle/problog cannot:

- **timesteps** — `Bob` becomes influential at *t=1*, `Carol` at *t=2*;
- **interval bounds** — derived `[0.8, 0.9]`, not boolean/point;
- **an event-log trace** with clause groundings (which node and which edge matched).

> First run JIT-compiles for ~170s; subsequent runs are ~8s.


In [19]:
from factgraph.adapters.pyreason.session import PyReasonSession
from factgraph.adapters.pyreason.runner import run_pyreason, PyReasonRunConfig
from factgraph.adapters.pyreason.rule_ext import compile_pyreason_rule, PyReasonRuleExt
from factgraph.sdk.dsl.expr import LogicVar, Pred
from factgraph.sdk.dsl.rule import Rule

schema_ir = {"predicates": [
    {"pred_id": "user:influential", "arity": 2},
    {"pred_id": "collab:weight", "arity": 3,
     "relationship_type": "Collab", "from_entity_type": "User", "to_entity_type": "User"},
]}
s = PyReasonSession(schema_ir)
s._write_node_fact_internal("user:influential", "Alice", "true", bound=[0.9, 0.9])   # seed
s._write_edge_fact_internal("collab:weight", "Alice", "Bob", "1")                    # Alice -> Bob
s._write_edge_fact_internal("collab:weight", "Bob", "Carol", "1")                    # Bob   -> Carol

x, y = LogicVar("x"), LogicVar("y")
rule = Rule(id="influence_spreads", version="1.0",
            select=[Pred("user:influential", x)],
            where=[Pred("user:influential", y), Pred("collab:weight", y, x)])
compiled = compile_pyreason_rule(rule, engine_ext=PyReasonRuleExt(
    timestep_delay=1, head_bound=(0.8, 0.9),
    body_predicate_bounds={"user:influential": (0.5, 1.0)},   # let the bounded seed match
))
print("PyReason rule:", compiled[0], "\n")

result = run_pyreason(s, rules=[compiled], config=PyReasonRunConfig(timesteps=3, atom_trace=True))
print("derived:", [(f["node_ref"], f["bound"], "from t=%d" % f["active_from"])
                   for f in result.derived_session.node_facts], "\n")
print("event-log trace:")
for e in result.trace_dict["node_events"]:
    if e["new_bound"] != [0.0, 1.0]:                          # skip 'unknown' rows
        due = e["occurred_due_to"]
        cl = (" via " + ", ".join(e["clause_groundings"])) if e["clause_groundings"] else ""
        print(f"  t={e['time']}  {e['component']:6} influential {e['new_bound']}  [{due}]{cl}")

PyReason rule: influential(x) : [0.8, 0.9] <-1 influential(y) : [0.5, 1.0], weight(y, x) 

torch is not installed, model integration is disabled


/Users/zhenzhili/miniforge3/envs/factpy/lib/python3.10/site-packages/pyreason/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


Added  0 graph-attribute node facts and  2 graph_attribute edge facts.
Filtering rules based on queries
Timestep: 0
Timestep: 1
Timestep: 2
Timestep: 3

Converged at time: 3
Fixed Point iterations: 4
derived: [('Bob', (0.8, 0.9), 'from t=1'), ('Carol', (0.8, 0.9), 'from t=2')] 

event-log trace:
  t=0  Alice  influential [0.9, 0.9]  [session_node_0]
  t=1  Alice  influential [0.9, 0.9]  [session_node_0]
  t=1  Bob    influential [0.8, 0.9]  [influence_spreads] via ['Alice'], [('Alice', 'Bob')]
  t=2  Alice  influential [0.9, 0.9]  [session_node_0]
  t=2  Bob    influential [0.8, 0.9]  [influence_spreads] via ['Alice', 'Bob'], [('Alice', 'Bob')]
  t=2  Carol  influential [0.8, 0.9]  [influence_spreads] via ['Alice', 'Bob'], [('Bob', 'Carol')]
  t=3  Alice  influential [0.9, 0.9]  [session_node_0]
  t=3  Bob    influential [0.8, 0.9]  [influence_spreads] via ['Alice', 'Bob'], [('Alice', 'Bob')]
  t=3  Carol  influential [0.8, 0.9]  [influence_spreads] via ['Alice', 'Bob'], [('Bob', 'Caro

## 9 · Summary — what each engine gives you

| Engine | Surface | Value domain | Time | Explanation shape | On these rules |
|---|---|---|---|---|---|
| `native` | SDK `evaluate` | boolean | — | reasoning chain | all base + composite rules |
| `problog` | SDK `evaluate` | point probability | — | chain **+ probabilities** (WMC) | all base + composite rules |
| `souffle` | SDK `evaluate` | boolean | — | proof tree | datalog-shaped bricks; not `colleagues` / large composites |
| `pyreason` | adapter runner | interval `[lo,hi]` | **timesteps** | event log + clause groundings | its own graph-propagation model |

**Composition takeaways**

- Build small, single-purpose **base rules**, then combine with `&` / `|` and port joins.
- A conclusion **head** (a tiny rule that exposes the output ports) names the result and carries its `repr`.
- Composition **narrows**: every pair here is a *colleague*, but only the project-sharing ones are *teammates*.
- Reuse a brick directly when its two uses fall in **separate OR branches**; use
  **per-side variants** when they share one conjunction.
- Author intent (probability, validity) lives in fact **meta**, independent of the
  engine — switching engines re-projects the same meta, it does not require rewriting facts.
- **Failure is explained, not hidden**: `evaluate` returns only holding rows, but a
  closed *explain head* (subjects pinned) lets `fg.eval.explain` report *why* a
  non-holding conclusion fails — `✗` / `○` on the exact atom, with the real value.
